- Download [dataset](https://www.kaggle.com/competitions/games-rating)
- put it in `Homework_11_neuro_regression/data/`

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
train_val_split = 0.8
learning_rate = 1e-3
batch_size = 64
epochs = 1000

device

device(type='mps')

In [21]:
train = pd.read_csv('./data/train_data.csv', index_col=0)
test = pd.read_csv('./data/test_data.csv', index_col=0)

test

,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
ID,,,,,,,,,,,,
161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
12333.0,Twilight Struggle,2005.0,2,2,180,13,40814,10,"3,59",56219.0,"Action/Event, Advantage Token, Area Majority /...","Strategy Games, Wargames"
115746.0,War of the Ring: Second Edition,2012.0,2,4,180,13,13725,12,"4,14",22281.0,"Area Majority / Influence, Area Movement, Camp...","Thematic Games, Wargames"
169786.0,Scythe,2016.0,1,5,115,14,57871,14,"3,41",75640.0,"Area Majority / Influence, Card Play Conflict ...",Strategy Games
28720.0,Brass: Lancashire,2007.0,2,4,120,14,19400,19,"3,86",25429.0,"Hand Management, Income, Loans, Network and Ro...",Strategy Games
...,...,...,...,...,...,...,...,...,...,...,...,...
6932.0,Hi Ho! Cherry-O,1960.0,2,4,10,3,1035,20325,"1,03",1691.0,"Cooperative Game, Roll / Spin and Move",Children's Games
3510.0,Battle of the Sexes,1997.0,2,8,45,12,1090,20328,"1,08",1987.0,Team-Based Game,Party Games
5895.0,Hungry Hungry Hippos,1978.0,2,4,10,4,2361,20330,"1,05",2568.0,NaN,Children's Games


In [22]:
from torch.utils.data import Dataset, DataLoader
from preprocessor import GameRatingPreprocessor

gpr = GameRatingPreprocessor()

train = gpr.fit_transform(train)
test = gpr.transform(test)


class TrainDataset(Dataset):
    def __init__(self, data: pd.DataFrame, label: str):
        # Keep labels shaped as (N, 1) so MSELoss matches model output (batch, 1)
        self.label = torch.tensor(data[label].to_numpy(), dtype=torch.float32).unsqueeze(1)
        self.data = torch.tensor(data.drop(columns=[label]).to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.label[idx]

class TestDataset(Dataset):
    def __init__(self, data: pd.DataFrame):
        self.data = torch.tensor(data.to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

train_val_dataset = TrainDataset(train, gpr.TARGET_COL)
train_dataset, val_dataset = torch.utils.data.random_split(train_val_dataset, [train_val_split, 1 - train_val_split])
test_dataset = TestDataset(test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [23]:
import torch.nn as nn
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        n_in = len(gpr.feature_columns_)
        self.layers = nn.Sequential(
            nn.Linear(n_in, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.layers(x)

net = Net()

criterion = nn.SmoothL1Loss()
optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)

In [24]:
from trainer import Trainer

trainer = Trainer(
    net,
    criterion,
    optimizer,
    device,
    epoch_amount=epochs,
    scheduler=lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=5
    ),
)
trainer.fit(train_loader, val_loader)

Эпоха: 0 Loss_train: 1.6484530732737785, 0:00:00.504775 сек
Loss_val: 0.20949274394661188

Эпоха: 1 Loss_train: 0.22066703743023397, 0:00:00.445653 сек
Loss_val: 0.16681753114486733

Эпоха: 2 Loss_train: 0.1882499976776033, 0:00:00.442689 сек
Loss_val: 0.11475090263411403

Эпоха: 3 Loss_train: 0.17343781447691442, 0:00:00.415770 сек
Loss_val: 0.10704776442920168

Эпоха: 4 Loss_train: 0.16071376454143624, 0:00:00.408064 сек
Loss_val: 0.10657487545783322

Эпоха: 5 Loss_train: 0.1438668907780922, 0:00:00.400633 сек
Loss_val: 0.10277071331317227

Эпоха: 6 Loss_train: 0.1413644188478667, 0:00:00.401895 сек
Loss_val: 0.10065535254155596

Эпоха: 7 Loss_train: 0.13506730755118176, 0:00:00.398768 сек
Loss_val: 0.08467354656507571

Эпоха: 8 Loss_train: 0.12950162023930026, 0:00:00.403308 сек
Loss_val: 0.07240932830609381

Эпоха: 9 Loss_train: 0.1260774496416147, 0:00:00.401785 сек
Loss_val: 0.10745580991109212

Эпоха: 10 Loss_train: 0.12307537943904936, 0:00:00.406468 сек
Loss_val: 0.07722431079

In [25]:
test_predictions = trainer.predict(test_loader)

pd.DataFrame(
    {"Rating Average": test_predictions.numpy().ravel()},
    index=pd.RangeIndex(len(test), dtype="int64"),
).to_csv("data/test_predictions.csv", index_label="index")